In [1]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weight_filename, naanobot_weight_filename,
                                          skeleton_config, naanobot_config, radial_config)

In [2]:
skeleton_weight_filename = skeleton_weight_filename
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weight_filename
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [4]:
drug_i_want_to_deliver = "CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C"

In [5]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C
Drug Likeness Rules Passed: 4 / 4


In [6]:
pocket_data = model.generate_pocket_data(temperature=1)
print(len(pocket_data))
print(type(pocket_data))

4
<class 'dict'>


In [7]:
pocket_data

{'SMILES': 'CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C',
 'Temperature': 1,
 '3D_skeleton': [[9.472589580972576, -11.751723627815567, -1.411084368894288],
  [13.071495007523337, 0.3010460330918045, -6.164681899872292],
  [8.566793100098668, -3.9509156256315277, -10.403812340817446],
  [-1.0937084594673416, -13.591657976489152, -0.07151627530523459],
  [1.5371313980348338, 12.6559858878589, -2.89821344571362],
  [0.8566575333366762, -10.345630135083338, -7.977107215427928],
  [9.998470673305544, 7.820018285510453, -2.638884538543015],
  [7.103556618514443, 10.102696111061336, -0.6157929208981828],
  [-3.012776469596022, 11.963024055204304, -2.0675220079523537],
  [3.1067533519992505, -11.908927135757086, 0.7049905157939624],
  [7.511515936298516, 4.616350352734873, -7.848996766362101],
  [8.16818366282858, 4.562004867965101, -7.075154872380581],
  [1.633576099269065, 9.35014174035117, -6.603049682590771],
  [3.2662433747932837, 10.209702161856251, -4.210615644987399],
  [7.07795

In [8]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [9]:
# GENERATION QUALITY ASSESSMENTS

In [10]:
import torch
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=10):
    model.eval()
    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(coord_context, bioch_context, coordinate).unsqueeze(0).to(map4_enc.device)
        output, _ = model.forward(naano_X, map4_enc)

        # print raw predicted vector
        print(f"\nPosition {i}:")
        print(f"  raw output: {output[0].detach().cpu().numpy().round(2)}")

        # print distances to all amino acids
        vector = output[0].detach().float()
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = torch.norm(vector - n_v_t).item()

        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        print(f"  top 3: {sorted_distances[:3]}")

        # update context
        aa_id = sorted_distances[0][0]
        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:] + [coordinate.tolist() if torch.is_tensor(coordinate) else coordinate]


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=model._pocket_sph_skeleton())


Position 0:
  raw output: [ 0.88 -0.4   0.19 -0.44  0.26 -0.19 -0.53  1.07  0.39 -1.02  0.8   0.24
  0.37  1.   -0.29]
  top 3: [('C', 1.95920729637146), ('M', 2.057156562805176), ('G', 2.1452994346618652)]

Position 1:
  raw output: [ 0.61 -0.26  0.59 -0.5   0.38  0.08 -0.63  0.86  0.23 -0.88  0.75  0.34
  0.34  0.86 -0.26]
  top 3: [('S', 1.850890040397644), ('C', 1.8517954349517822), ('T', 1.8998527526855469)]

Position 2:
  raw output: [ 0.59 -0.34  0.22 -0.47  0.38  0.11 -0.43  0.89  0.41 -1.02  0.78  0.4
  0.39  1.02 -0.47]
  top 3: [('C', 1.9890480041503906), ('N', 2.020195484161377), ('S', 2.03507399559021)]

Position 3:
  raw output: [ 0.45 -0.23  0.37 -0.49  0.44  0.21 -0.49  0.75  0.05 -0.91  0.89  0.4
  0.38  0.97 -0.25]
  top 3: [('S', 1.8049269914627075), ('N', 1.8137949705123901), ('T', 1.8632066249847412)]

Position 4:
  raw output: [ 0.64 -0.31  0.31 -0.41  0.24  0.21 -0.61  0.93  0.42 -0.99  0.59  0.45
  0.4   1.02 -0.32]
  top 3: [('C', 1.8702490329742432), ('S', 1.